# Usage examples 
The files in `examples/data` are based on data sets from [Liander open data](https://www.liander.nl/over-ons/open-data), 
specifically "Verbruiksdata slimme meter 2012-2014".

This example requires `hvplot` to be installed for plotting.

In [1]:
from pathlib import Path

import hvplot.pandas  # noqa: F401
import pandas as pd

from sundael import PVConfig, PVDisaggregator

data_dir = Path("./data")

## Basic example without extra configuration

In [2]:
pv_ratio = pd.read_parquet(data_dir / "pv_ratio.parquet")
net_con = pd.read_parquet(data_dir / "net_con.parquet").T
net_gen = pd.read_parquet(data_dir / "net_gen.parquet").T

In [3]:
pv_ratio.sample(3)

,average
datetime,
2013-03-25 21:00:00+00:00,0.000000
2013-11-13 13:45:00+00:00,0.371401
2013-11-19 18:15:00+00:00,0.000000


In [4]:
net_con.sample(3).iloc[:, :5]

datetime,2013-01-01 00:00:00+01:00,2013-01-01 00:15:00+01:00,2013-01-01 00:30:00+01:00,2013-01-01 00:45:00+01:00,2013-01-01 01:00:00+01:00
Klant 69,107.0,98.0,91.0,88.0,89.0
Klant 79,223.0,257.0,179.0,216.0,209.0
Klant 21,24.0,21.0,13.0,13.0,41.0


In [5]:
net_gen.sample(3).iloc[:, :5]

datetime,2013-01-01 00:00:00+01:00,2013-01-01 00:15:00+01:00,2013-01-01 00:30:00+01:00,2013-01-01 00:45:00+01:00,2013-01-01 01:00:00+01:00
Klant 1,0,0,0,0,0
Klant 79,0,0,0,0,0
Klant 21,0,0,0,0,0


In [6]:
# Note: mapping is optional if only one reference (area) is present
area_mapping = {k: "average" for k in net_con.index}
pv = PVDisaggregator(pv_ratio=pv_ratio, area_mapping=area_mapping)

result = pv.disaggregate(net_con, net_gen)

2026-09-11 07:12:52 [info     ] Setting temperature based on data from the Dutch meteorological institute.
2026-09-11 07:12:52 [info     ] This is not valid for other localities! Set auto_infer_temp to `False` 
2026-09-11 07:12:52 [info     ] to disable or provide temperature to PVDisaggregator.
________________________________________________________________________________
[Memory] Calling sundael.temperature._get_knmi_hourly_temperature_series...
_get_knmi_hourly_temperature_series('2013010100', '2013123123')
2026-09-11 07:12:52 [info     ] Accessing KNMI API with period 201301010000 - 201312312300
_______________________________get_knmi_hourly_temperature_series - 1.8s, 0.0min
2026-09-11 07:12:54 [info     ] Optimizing parameters         
2026-09-11 07:12:54 [info     ]   - Create parameter array    
2026-09-11 07:12:55 [info     ]   - Optimization              
2026-09-11 07:12:59 [info     ] Estimating generation...      
2026-09-11 07:12:59 [info     ]   - Area-dependent paramet

In [7]:
# Show one example of "Klant 63"

idx = "Klant 63"

pd.concat(
    (
        result["consumption"].loc[idx].rename("consumption"),
        result["generation"].loc[idx].rename("generation") * -1,
    ),
    axis=1,
).fillna(0).hvplot(width=1000, title=idx)

:NdOverlay   [Variable]
   :Curve   [datetime]   (value)

## Examples with custom configuration

The previous example assumes the data originates from the Netherlands. However, the code will
also work for other localities. In this case, you will have to provide location (latitude and
longitude) and the temperature. 

Note that the timezone of the `DateTimeIndex` should be set (it will be converted to UTC internally).

The most straightforward way is to provide a default via the config. In this case, the `auto_infer_temp` 
value should be set to `False`. 

In [ ]:
# Example values for the Netherlands
config = PVConfig(latitude=52.3, longitude=4.7, yearly_avg_temp=11.3, auto_infer_temp=False)

pv = PVDisaggregator(pv_ratio=pv_ratio, config=config)

result = pv.disaggregate(net_con, net_gen)

2026-09-11 07:13:01 [info     ] Optimizing parameters         
2026-09-11 07:13:01 [info     ]   - Create parameter array    
2026-09-11 07:13:01 [info     ]   - Optimization              
2026-09-11 07:13:05 [info     ] Estimating generation...      
2026-09-11 07:13:05 [info     ]   - Area-dependent parameters 
2026-09-11 07:13:06 [info     ]   - Customer-dependent generation
2026-09-11 07:13:06 [info     ]   - Final calculation         
2026-09-11 07:13:06 [info     ] Done estimating generation    
2026-09-11 07:13:06 [info     ] Calculation generation and consumption
2026-09-11 07:13:06 [info     ] Calculate self consumption    
2026-09-11 07:13:06 [info     ] Calculate rule-based self consumption
2026-09-11 07:13:06 [info     ] Calculate simple self consumption
2026-09-11 07:13:06 [info     ] Done with disaggregation      


You can also set the latitude and longitude specifically for multiple geographical areas. The
following example shows how to do this for a synthetic data set. In reality, you would provide
a PV ratio for each area, with an accompanying latitude and longitude.

In [ ]:
# Note: this is fake data. The Liander data set is anonymized and contains no geographic information!

# Latitude and longitude for each area (Dutch cities in this synthetic example)
area_latlon = pd.DataFrame(
    {
        "latitude": [52.3676, 51.9225, 52.0705, 52.0902, 51.4416, 53.2129, 50.8308, 52.3667, 51.5597, 51.5897],
        "longitude": [4.9041, 4.4792, 4.3007, 5.1075, 5.4697, 6.5795, 5.6915, 5.1833, 5.0919, 4.7789],
    },
    index=[
        "Amsterdam",
        "Rotterdam",
        "Den Haag",
        "Utrecht",
        "Eindhoven",
        "Groningen",
        "Maastricht",
        "Almere",
        "Tilburg",
        "Breda",
    ],
)

# A separate PV ratio for each area.
pv_ratio_area = pd.concat([pv_ratio.rename(columns={"average": area}) for area in area_latlon.index], axis=1)

# Mapping of customers / identifiers to areas.
area_mapping = dict(zip(net_con.index, area_latlon.index, strict=True))

In [ ]:
# We still set the yearly average temperature
config = PVConfig(yearly_avg_temp=11.3, auto_infer_temp=False)

# Area-related data is provided at initialization
pv = PVDisaggregator(pv_ratio=pv_ratio_area, config=config, area_mapping=area_mapping, area_latlon=area_latlon)

result = pv.disaggregate(net_con, net_gen)

2026-06-30 09:48:21 [info     ] Optimizing parameters         
2026-06-30 09:48:21 [info     ]   - Create parameter array    


2026-06-30 09:48:21 [info     ]   - Optimization              
2026-06-30 09:48:25 [info     ] Estimating generation...      
2026-06-30 09:48:25 [info     ]   - Area-dependent parameters 
2026-06-30 09:48:27 [info     ]   - Customer-dependent generation
2026-06-30 09:48:27 [info     ]   - Final calculation         
2026-06-30 09:48:27 [info     ] Done estimating generation    
2026-06-30 09:48:27 [info     ] Calculation generation and consumption
2026-06-30 09:48:27 [info     ] Calculate self consumption    
2026-06-30 09:48:27 [info     ] Calculate rule-based self consumption
2026-06-30 09:48:27 [info     ] Calculate simple self consumption
2026-06-30 09:48:27 [info     ] Done with disaggregation      


Finally, instead of setting the an average temperature, you can also provide the temperature per time-point.

In [ ]:
# Provide temperature (in th)
temperature = pd.Series(
    # This just an example to demonstrate the functionality, provide real values here!
    [11.3] * len(pv_ratio.index),
    # Time points should be the same!
    index=pv_ratio.index,
)

# We still set the yearly average temperature
config = PVConfig(auto_infer_temp=False)

pv = PVDisaggregator(pv_ratio=pv_ratio, temperature=temperature, config=config)

result = pv.disaggregate(net_con, net_gen)

2026-06-30 09:48:27 [info     ] Optimizing parameters         
2026-06-30 09:48:27 [info     ]   - Create parameter array    
2026-06-30 09:48:27 [info     ]   - Optimization              
2026-06-30 09:48:31 [info     ] Estimating generation...      
2026-06-30 09:48:31 [info     ]   - Area-dependent parameters 
2026-06-30 09:48:32 [info     ]   - Customer-dependent generation
2026-06-30 09:48:32 [info     ]   - Final calculation         
2026-06-30 09:48:32 [info     ] Done estimating generation    
2026-06-30 09:48:32 [info     ] Calculation generation and consumption
2026-06-30 09:48:32 [info     ] Calculate self consumption    
2026-06-30 09:48:32 [info     ] Calculate rule-based self consumption
2026-06-30 09:48:32 [info     ] Calculate simple self consumption
2026-06-30 09:48:32 [info     ] Done with disaggregation      
